# Safety projection on Colab

Run the cells in order. The plan comes from the To_Do file(s) named below, in task-id order. To run two sessions at once, give each its own `SHARD` (0 and 1) and set `NUM_SHARDS = 2` in both.

In [ ]:
!pip install -q "git+https://github.com/AAAA477/orthovac-safety.git"

In [ ]:
#@title Settings { display-mode: "form" }
SIDE = 'Qwen'
TODO_PATHS = ['/content/drive/MyDrive/Safety-projections/Qwen/Qwen_Experiment_To_Do.json']
SHARD, NUM_SHARDS = 0, 1
RUN_UPLOAD = True

In [ ]:
import orthovac as ov

cfg = ov.setup(side=SIDE, todo_paths=TODO_PATHS, shard=SHARD, num_shards=NUM_SHARDS)
ov.harden_judge()          # judge retries; a quota failure stops the run with a clear message
plan = ov.show_plan(cfg)   # the verified To_Do read and the ordered plan

In [ ]:
ov.build_adapters(cfg, plan)
if RUN_UPLOAD:
    ov.upload_adapters(cfg, plan)

In [ ]:
# Gate: one evaluation, then check the output has exactly the two score columns.
from orthovac.judge import score_columns

await ov.run_evals(cfg, plan, limit=1)
first = sorted(cfg.runs_root.rglob('*_eval-*.csv'))[0]
print(first.name, '->', score_columns(first))   # expect ['aligned', 'coherent']

In [ ]:
# Every evaluation is saved (verified Drive copy, summary row, log line) before the next one starts.
waiting = await ov.run_evals(cfg, plan)

In [ ]:
# Run once, after every shard has finished.
ov.merge_summaries(cfg)
ov.sync_status(cfg, plan)

In [ ]:
ov.line_graphs(cfg)       # PNG + editable SVG under runs/_figures

In [ ]:
found, new_ok, new_bad = ov.find_results(cfg)   # anything on disk the summaries do not know yet